# 05 SciBERT SolarPhysics Search - rebuild

Notebook canonico da Camada 1 para treinar novamente o retriever especializado
`SciBERT_SolarPhysics_Search` usando **apenas**:

- `Nucleo_core`
- `PIML_core`
- `CombFinal_core`

## Papel metodologico

Este notebook:

1. consome os corpora canonicos do `00_consolidacao`;
2. consome os artefatos semanticos do `01_abstract_llm` para auditoria de insumos;
3. monta o corpus textual de dominio e os pares fracos de contraste;
4. executa `DAPT` e depois `contrastive fine-tuning`;
5. salva checkpoints, manifestos, relatorios e artefatos no Google Drive.

## Regra metodologica

- `ML_Multimodal` **nao** entra no treino.
- O treino permanece restrito ao `core`.
- O notebook precisa ser acompanhado por prints frequentes e por logs salvos no Drive.


In [ ]:
# ============================================================
# Instalacao de dependencias para Colab GPU
# ============================================================
!pip install -U -q pip setuptools wheel
!pip install -U -q numpy==2.1.2 scipy==1.14.1 pandas==2.2.2 scikit-learn==1.5.2
!pip install -U -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -U -q transformers==4.45.2 datasets==3.0.1 accelerate==0.34.2 sentence-transformers==2.7.0
!pip install -U -q pyarrow openpyxl pyreadr faiss-cpu

print("Dependencias instaladas.")


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
import math
import os
import random
import re
import shutil
import time
import unicodedata
import warnings
from datetime import datetime
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sentence_transformers import InputExample, SentenceTransformer, losses
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForMaskedLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 180)
pd.set_option("display.max_colwidth", 240)

print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# Configuracao geral
# ============================================================

DRIVE_ROOT = Path("/content/drive/MyDrive/Unicamp")
PROJECT_ROOT = DRIVE_ROOT / "artigo bibliometria" / "grounded-scientometrics-solarphysics-retrieval"
DATA_ROOT = DRIVE_ROOT / "artigo bibliometria" / "base de dados" / "Artigo_Bibliometria Base Bruta" / "BASES_UNIFICADAS_POR_TEMA"

READ_STAGE_CONSOL = "00_consolidacao"
READ_STAGE_ABSTRACT = "01_abstract_llm"
TRAIN_CORPORA = ["Nucleo", "PIML", "CombFinal"]
WRITE_ROOT = DATA_ROOT / "_cross_corpus_rebuild" / "05_scibert_solarphysics_search"
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

BASE_MODEL_ID = "allenai/scibert_scivocab_uncased"
MAX_SEQ_LEN = 256
MIN_TOKENS = 16
MAX_DAPT_DOCS = None
MAX_CONTRASTIVE_PAIRS = 120000
RUN_DAPT = True
RUN_CONTRASTIVE = True
FORCE_RETRAIN = False
DAPT_EPOCHS = 1
CONTRASTIVE_EPOCHS = 1
DAPT_BATCH_SIZE = 8
CONTRASTIVE_BATCH_SIZE = 32
TOPICS_PER_CORPUS_FOR_AUDIT = 12

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT nao encontrado: {PROJECT_ROOT}"
assert DATA_ROOT.exists(), f"DATA_ROOT nao encontrado: {DATA_ROOT}"

print("WRITE_ROOT =", WRITE_ROOT)
print("TRAIN_CORPORA =", TRAIN_CORPORA)
print("BASE_MODEL_ID =", BASE_MODEL_ID)
print("RUN_DAPT =", RUN_DAPT)
print("RUN_CONTRASTIVE =", RUN_CONTRASTIVE)


In [ ]:
# ============================================================
# Saidas, logs e helpers
# ============================================================

PIPE_START_TS = time.time()


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


ARTIFACTS_DIR = ensure_dir(WRITE_ROOT / "artifacts")
REPORTS_DIR = ensure_dir(WRITE_ROOT / "reports")
CHECKPOINT_DIR = ensure_dir(WRITE_ROOT / "checkpoints")
DAPT_DIR = ensure_dir(CHECKPOINT_DIR / "scibert_dapt")
FINAL_MODEL_DIR = ensure_dir(CHECKPOINT_DIR / "SciBERT-SolarPhysics-Search")
GLOBAL_LOG_DIR = ensure_dir(PROJECT_ROOT / "outputs" / "camada2_logs")
GLOBAL_LOG_FILE = GLOBAL_LOG_DIR / f"05_scibert_rebuild_{RUN_TS}.txt"


def elapsed_seconds() -> float:
    return time.time() - PIPE_START_TS


def fmt_seconds(seconds: float) -> str:
    seconds = int(round(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"


def log(message: str) -> None:
    now = datetime.now().strftime("%H:%M:%S")
    prefix = f"[{now} | +{fmt_seconds(elapsed_seconds())}]"
    line = f"{prefix} {message}"
    print(line, flush=True)
    with open(GLOBAL_LOG_FILE, "a", encoding="utf-8") as fh:
        fh.write(line + "\n")


def stage_banner(title: str) -> None:
    bar = "=" * 96
    log(bar)
    log(title)
    log(bar)


def read_consol_csv(corpus: str) -> Path:
    return DATA_ROOT / corpus / "04_rebuild_outputs" / READ_STAGE_CONSOL / f"{corpus}_core_bibliometrix_clean.csv"


def read_topic_cards_csv(corpus: str) -> Path:
    return DATA_ROOT / corpus / "04_rebuild_outputs" / READ_STAGE_ABSTRACT / "core" / "tables" / f"{corpus}_core_topic_cards.csv"


print("GLOBAL_LOG_FILE =", GLOBAL_LOG_FILE)
print("FINAL_MODEL_DIR =", FINAL_MODEL_DIR)


In [ ]:
# ============================================================
# Leitura dos corpora de treino e auditoria dos insumos
# ============================================================

EXPECTED_COLS = ["TI", "AB", "DE", "ID", "SO", "PY", "TC", "DI"]


def clean_text(value):
    if value is None:
        return pd.NA
    try:
        if pd.isna(value):
            return pd.NA
    except Exception:
        pass
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "<na>"}:
        return pd.NA
    return text


def normalize_semantic_text(value):
    text = clean_text(value)
    if pd.isna(text):
        return pd.NA
    text = unicodedata.normalize("NFKC", str(text)).lower()
    text = re.sub(r"http\S+|www\.\S+|\b10\.\d{4,9}/\S+\b", " ", text)
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return pd.NA
    return text


def ensure_expected_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in EXPECTED_COLS:
        if col not in out.columns:
            out[col] = pd.NA
    return out


def prepare_train_frame(corpus: str) -> pd.DataFrame:
    csv_path = read_consol_csv(corpus)
    assert csv_path.exists(), f"Input nao encontrado: {csv_path}"
    raw = pd.read_csv(csv_path, dtype=str, low_memory=False)
    raw = ensure_expected_cols(raw)
    raw["corpus"] = corpus
    raw["title_clean"] = raw["TI"].map(normalize_semantic_text)
    raw["abstract_clean"] = raw["AB"].map(normalize_semantic_text)
    raw["keywords_clean"] = (
        raw["DE"].fillna("").astype(str) + "; " + raw["ID"].fillna("").astype(str)
    ).map(normalize_semantic_text)
    raw["text_full"] = (
        raw["title_clean"].fillna("").astype(str)
        + ". "
        + raw["abstract_clean"].fillna("").astype(str)
        + ". keywords: "
        + raw["keywords_clean"].fillna("").astype(str)
    ).str.replace(r"\s+", " ", regex=True).str.strip()
    raw["query_side"] = (
        raw["title_clean"].fillna("").astype(str)
        + ". keywords: "
        + raw["keywords_clean"].fillna("").astype(str)
    ).str.replace(r"\s+", " ", regex=True).str.strip()
    raw["positive_side"] = raw["abstract_clean"].fillna(raw["text_full"])
    raw["text_tokens"] = raw["text_full"].fillna("").astype(str).str.split().str.len()
    usable = raw[raw["text_tokens"] >= MIN_TOKENS].copy().reset_index(drop=True)
    usable["doc_local_id"] = [f"{corpus}_{i:07d}" for i in range(len(usable))]
    log(
        f"[load] {corpus} | input={len(raw)} | usable={len(usable)} | "
        f"abstract_pct={round(raw['abstract_clean'].notna().mean() * 100, 2) if len(raw) else 0.0}%"
    )
    return usable


stage_banner("LEITURA DOS CORPORA DE TREINO")

domain_frames = []
audit_rows = []
for corpus in TRAIN_CORPORA:
    df = prepare_train_frame(corpus)
    domain_frames.append(df)
    audit_rows.append(
        {
            "corpus": corpus,
            "rows_usable": int(len(df)),
            "mean_tokens": round(float(df["text_tokens"].mean()), 2) if len(df) else 0.0,
            "mean_citations": round(float(pd.to_numeric(df["TC"], errors="coerce").fillna(0).mean()), 2) if len(df) else 0.0,
        }
    )
    cards_path = read_topic_cards_csv(corpus)
    if cards_path.exists():
        cards = pd.read_csv(cards_path)
        log(f"[audit] {corpus} | topic_cards_core={len(cards)} | path={cards_path}")
        display(cards.head(TOPICS_PER_CORPUS_FOR_AUDIT))
    else:
        log(f"[audit] {corpus} | topic_cards_core ausente em {cards_path}")

train_df = pd.concat(domain_frames, ignore_index=True)
if MAX_DAPT_DOCS:
    train_df = train_df.head(MAX_DAPT_DOCS).copy()

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(ARTIFACTS_DIR / "domain_core_corpus_audit.csv", index=False)
train_df.to_csv(ARTIFACTS_DIR / "domain_core_training_corpus.csv", index=False)

log(f"[load] train_df total={len(train_df)}")
display(audit_df)
display(train_df.head(3))


In [ ]:
# ============================================================
# Montagem do corpus DAPT e dos pares fracos de contraste
# ============================================================

stage_banner("MONTAGEM DE INSUMOS DE TREINO")

dapt_texts = train_df["text_full"].dropna().astype(str).unique().tolist()
pair_df = train_df[["corpus", "doc_local_id", "query_side", "positive_side", "PY", "DI", "SO"]].copy()
pair_df = pair_df[
    pair_df["query_side"].fillna("").astype(str).str.split().str.len().ge(4)
    & pair_df["positive_side"].fillna("").astype(str).str.split().str.len().ge(8)
].copy()
pair_df = pair_df.drop_duplicates(subset=["query_side", "positive_side"]).reset_index(drop=True)

if MAX_CONTRASTIVE_PAIRS and len(pair_df) > MAX_CONTRASTIVE_PAIRS:
    pair_df = pair_df.sample(MAX_CONTRASTIVE_PAIRS, random_state=42).reset_index(drop=True)

pair_df["query_tokens"] = pair_df["query_side"].fillna("").astype(str).str.split().str.len()
pair_df["positive_tokens"] = pair_df["positive_side"].fillna("").astype(str).str.split().str.len()

log(f"[pairs] dapt_texts={len(dapt_texts)}")
log(f"[pairs] contrastive_pairs={len(pair_df)}")

pair_df.to_csv(ARTIFACTS_DIR / "contrastive_pairs.csv", index=False)
pd.DataFrame(
    [
        {
            "n_dapt_texts": len(dapt_texts),
            "n_pairs": len(pair_df),
            "mean_query_tokens": round(float(pair_df["query_tokens"].mean()), 2) if len(pair_df) else 0.0,
            "mean_positive_tokens": round(float(pair_df["positive_tokens"].mean()), 2) if len(pair_df) else 0.0,
        }
    ]
).to_csv(ARTIFACTS_DIR / "training_input_summary.csv", index=False)

display(pair_df.head(5))


In [ ]:
# ============================================================
# DAPT (Masked Language Modeling) no core de dominio
# ============================================================

stage_banner("DAPT - SCIBERT")

if FINAL_MODEL_DIR.exists() and any(FINAL_MODEL_DIR.iterdir()) and not FORCE_RETRAIN:
    log(f"[dapt] modelo final ja existe em {FINAL_MODEL_DIR}. Reuso sem retreino.")
    dapt_base_path = str(FINAL_MODEL_DIR)
else:
    if RUN_DAPT:
        raw_ds = Dataset.from_dict({"text": dapt_texts})
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
        mlm_model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL_ID)

        def tokenize_batch(batch):
            return tokenizer(
                batch["text"],
                truncation=True,
                max_length=MAX_SEQ_LEN,
                padding="max_length",
            )

        tokenized_ds = raw_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
        collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

        args = TrainingArguments(
            output_dir=str(DAPT_DIR),
            overwrite_output_dir=True,
            num_train_epochs=DAPT_EPOCHS,
            per_device_train_batch_size=DAPT_BATCH_SIZE,
            logging_steps=25,
            save_steps=250,
            save_total_limit=2,
            report_to=[],
            fp16=torch.cuda.is_available(),
        )

        trainer = Trainer(
            model=mlm_model,
            args=args,
            train_dataset=tokenized_ds,
            data_collator=collator,
        )

        log(f"[dapt] iniciando treino MLM | docs={len(dapt_texts)} | epochs={DAPT_EPOCHS}")
        trainer.train()
        trainer.save_model(str(DAPT_DIR))
        tokenizer.save_pretrained(str(DAPT_DIR))
        with open(REPORTS_DIR / "dapt_training_args.json", "w", encoding="utf-8") as fh:
            json.dump(args.to_dict(), fh, indent=2, ensure_ascii=False)
        dapt_base_path = str(DAPT_DIR)
        log(f"[dapt] concluido | checkpoint={DAPT_DIR}")
    else:
        dapt_base_path = BASE_MODEL_ID
        log("[dapt] desativado por configuracao. Seguindo direto para o contrastivo.")

print("Base para o contrastivo =", dapt_base_path)


In [ ]:
# ============================================================
# Fine-tuning contrastivo (Sentence-Transformers)
# ============================================================

stage_banner("FINE-TUNING CONTRASTIVO")

if FINAL_MODEL_DIR.exists() and any(FINAL_MODEL_DIR.iterdir()) and not FORCE_RETRAIN:
    log(f"[contrastive] modelo final ja existe em {FINAL_MODEL_DIR}. Reuso sem retreino.")
else:
    if not RUN_CONTRASTIVE:
        raise RuntimeError("RUN_CONTRASTIVE=False e nao existe modelo final pronto.")

    base_for_sentence = dapt_base_path if isinstance(dapt_base_path, str) else BASE_MODEL_ID
    st_model = SentenceTransformer(base_for_sentence, device="cuda" if torch.cuda.is_available() else "cpu")
    st_model.max_seq_length = MAX_SEQ_LEN

    pair_examples = [
        InputExample(texts=[row.query_side, row.positive_side])
        for row in pair_df.itertuples(index=False)
    ]
    train_loader = DataLoader(pair_examples, shuffle=True, batch_size=CONTRASTIVE_BATCH_SIZE)
    train_loss = losses.MultipleNegativesRankingLoss(st_model)
    warmup_steps = max(10, int(len(train_loader) * CONTRASTIVE_EPOCHS * 0.1))

    log(
        f"[contrastive] examples={len(pair_examples)} | batches={len(train_loader)} | "
        f"epochs={CONTRASTIVE_EPOCHS} | warmup={warmup_steps}"
    )

    st_model.fit(
        train_objectives=[(train_loader, train_loss)],
        epochs=CONTRASTIVE_EPOCHS,
        warmup_steps=warmup_steps,
        output_path=str(FINAL_MODEL_DIR),
        show_progress_bar=True,
    )

    log(f"[contrastive] concluido | final_model={FINAL_MODEL_DIR}")

assert FINAL_MODEL_DIR.exists(), f"Modelo final nao encontrado: {FINAL_MODEL_DIR}"


In [ ]:
# ============================================================
# Auditoria final, artefatos e manifesto
# ============================================================

stage_banner("AUDITORIA FINAL DO RETRIEVER")

final_model = SentenceTransformer(str(FINAL_MODEL_DIR), device="cuda" if torch.cuda.is_available() else "cpu")
sample_df = train_df.groupby("corpus", group_keys=False).head(400).copy()
sample_texts = sample_df["text_full"].fillna("").astype(str).tolist()
sample_emb = final_model.encode(sample_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True).astype("float32")
faiss.normalize_L2(sample_emb)

weak_query_rows = []
for corpus in TRAIN_CORPORA:
    cards_path = read_topic_cards_csv(corpus)
    if not cards_path.exists():
        continue
    cards = pd.read_csv(cards_path).head(TOPICS_PER_CORPUS_FOR_AUDIT)
    for row in cards.itertuples(index=False):
        query_text = f"{getattr(row, 'title', '')}. {getattr(row, 'keywords', '')}".strip()
        if len(query_text.split()) < 3:
            continue
        q_emb = final_model.encode([query_text], convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(q_emb)
        scores = np.matmul(sample_emb, q_emb[0])
        top_idx = np.argsort(scores)[::-1][:5]
        for rank, idx in enumerate(top_idx, start=1):
            hit = sample_df.iloc[int(idx)]
            weak_query_rows.append(
                {
                    "source_corpus": corpus,
                    "query_text": query_text,
                    "rank": rank,
                    "score": float(scores[idx]),
                    "hit_corpus": hit["corpus"],
                    "hit_title": hit["TI"],
                    "hit_year": hit["PY"],
                    "hit_doi": hit["DI"],
                }
            )

weak_query_df = pd.DataFrame(weak_query_rows)
weak_query_df.to_csv(ARTIFACTS_DIR / "weak_query_neighbors.csv", index=False)

manifest_rows = []
for path in sorted(WRITE_ROOT.rglob("*")):
    if path.is_file():
        manifest_rows.append(
            {
                "artifact": str(path),
                "size_bytes": path.stat().st_size,
            }
        )

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(REPORTS_DIR / "artifact_manifest.csv", index=False)

training_manifest = {
    "train_corpora": TRAIN_CORPORA,
    "n_train_docs": int(len(train_df)),
    "n_dapt_texts": int(len(dapt_texts)),
    "n_contrastive_pairs": int(len(pair_df)),
    "base_model_id": BASE_MODEL_ID,
    "run_dapt": RUN_DAPT,
    "run_contrastive": RUN_CONTRASTIVE,
    "run_ts": RUN_TS,
}
with open(REPORTS_DIR / "training_manifest.json", "w", encoding="utf-8") as fh:
    json.dump(training_manifest, fh, indent=2, ensure_ascii=False)

display(pd.read_csv(ARTIFACTS_DIR / "training_input_summary.csv"))
display(weak_query_df.head(20))
display(manifest_df.head(30))

print("Artefatos finais salvos em:", WRITE_ROOT)
